# Day 058 — Exercise 4: WebSocket Echo App

**WebSockets** provide a full-duplex channel: both client and server can send messages at any time. Unlike SSE (one-way), WebSockets are bidirectional.

`@app.websocket('/ws')` + `async def endpoint(ws: WebSocket):` is the FastAPI pattern. `TestClient.websocket_connect()` lets you test without a real browser — no async event loop setup needed on your side.

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from starlette.testclient import TestClient


## Task

Implement `build_ws_app()` — return a FastAPI app with:

```
WebSocket /ws
```

1. `await ws.accept()` — complete the WebSocket handshake
2. Loop: `data = await ws.receive_text()` → `await ws.send_text(f'Echo: {data}')`
3. `except WebSocketDisconnect: pass` — exit cleanly when client closes

## Your Implementation

In [ ]:
def build_ws_app() -> FastAPI:
    """Build a FastAPI app with a WebSocket endpoint at /ws.

    The endpoint should:
    1. Accept the connection: await ws.accept()
    2. Loop: receive text, send back 'Echo: {message}'
    3. Exit cleanly on WebSocketDisconnect
    """
    # TODO: build app with @app.websocket("/ws") async def endpoint
    raise NotImplementedError


In [ ]:
def build_ws_app() -> FastAPI:
    app = FastAPI()

    @app.websocket("/ws")
    async def ws_endpoint(ws: WebSocket):
        await ws.accept()
        try:
            while True:
                data = await ws.receive_text()
                await ws.send_text(f"Echo: {data}")
        except WebSocketDisconnect:
            pass

    return app


## Automated checks

In [ ]:
score, total = 0, 3
try:
    app    = build_ws_app()
    client = TestClient(app, raise_server_exceptions=False)

    with client.websocket_connect("/ws") as ws:
        score += 1; print("\u2705 /ws accepts WebSocket connection")

        ws.send_text("hello")
        data = ws.receive_text()
        assert data == "Echo: hello", f"Expected 'Echo: hello', got {data!r}"
        score += 1; print("\u2705 echoes first message with 'Echo: ' prefix")

        ws.send_text("world")
        data2 = ws.receive_text()
        assert data2 == "Echo: world", f"Expected 'Echo: world', got {data2!r}"
        score += 1; print("\u2705 echoes second message correctly")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_ws_app() -> FastAPI:
    app = FastAPI()

    @app.websocket("/ws")
    async def ws_endpoint(ws: WebSocket):
        await ws.accept()
        try:
            while True:
                data = await ws.receive_text()
                await ws.send_text(f"Echo: {data}")
        except WebSocketDisconnect:
            pass

    return app
```

**Why it works:** The `with client.websocket_connect('/ws')` context manager opens the handshake and closes it on exit, which triggers `WebSocketDisconnect` in the server — that is why we catch it instead of letting it propagate.

</details>